# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjoyy/ml-flyrankai/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing **Lane 4: CTR / Engagement Opportunity Scoring**.

This lane asks: *Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?* It's the right choice because the data shows a massive gap between where pages rank and how often they get clicked. With 14,305 pages sitting on page 1-2 with CTR below 1%, there's a clear, measurable opportunity to improve traffic without improving rankings. The decision is concrete—which pages need title/meta description rewrites—and the cost of inaction is quantifiable in lost clicks.

In [ ]:
import pandas as pd
import numpy as np
import os

# Find the data file relative to the repo root
# In Colab/GitHub, CWD is the repo root; locally, navigate up from work/notebooks/
candidates = [
    os.path.join('data', 'raw', 'content_refresh_anonymized.csv'),
    os.path.join('..', '..', 'data', 'raw', 'content_refresh_anonymized.csv'),
]
data_path = next(p for p in candidates if os.path.exists(p))

df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.shape[1]}')
print(f'\nContent types:')
print(df['content_type'].value_counts())
print(f'\nPosition tiers:')
print(df['position_tier'].value_counts())

## 2. The question: decision, action, cost of a wrong call

**Decision:** Which pages should the content team prioritize for title, meta description, or snippet improvements to increase organic click-through rate?

**Action:** The content team reviews the top-ranked pages and rewrites their titles and meta descriptions to better match search intent and stand out in SERPs.

**Cost of a wrong call:**
- **False positive (recommending a page that doesn't need work):** Wasted editorial time—someone spends 30-60 minutes rewriting a title that was already fine. This is low-cost per instance but adds up if the model is wrong often.
- **False negative (missing a page that should be improved):** Lost clicks and traffic. A page sitting at position 5 with 0.2% CTR when it could have 2% means missing 90% of potential visitors. For high-impression pages, this is significant revenue or lead opportunity cost.

In [ ]:
# Focus on pages with enough impressions to matter (>= 100)
visible = df[df['impressions_90d'] >= 100].copy()

print(f'Pages with >= 100 impressions: {len(visible)} ({len(visible)/len(df)*100:.1f}% of total)')
print(f'\nCTR distribution for visible pages:')
print(f'  Mean CTR: {visible["ctr"].mean():.3f}%')
print(f'  Median CTR: {visible["ctr"].median():.3f}%')
print(f'  Pages with CTR < 1%: {(visible["ctr"] < 1).sum()} ({(visible["ctr"] < 1).mean()*100:.1f}%)')

print(f'\nPosition distribution for visible pages:')
print(f'  Mean position: {visible["avg_position"].mean():.1f}')
print(f'  Pages on page 1 (position <= 10): {(visible["avg_position"] <= 10).sum()} ({(visible["avg_position"] <= 10).mean()*100:.1f}%)')
print(f'  Pages on page 2-3 (position 11-30): {((visible["avg_position"] > 10) & (visible["avg_position"] <= 30)).sum()} ({((visible["avg_position"] > 10) & (visible["avg_position"] <= 30)).mean()*100:.1f}%)')

## 3. Quick look at the data (2-3 real numbers)

Here are three numbers that make this lane worth pursuing:

1. **14,305 pages** have position ≤ 20 but CTR < 1%. These pages are visible in search results but barely getting clicked—a massive untapped opportunity.
2. **113.9 million impressions** flow through those low-CTR pages. If we can help raise their CTR even modestly, the click gain is substantial.
3. **7,166 page-1 pages** (position ≤ 10) have CTR < 0.5%. These are the highest-value candidates—they're already ranking well but their snippets aren't converting searches into clicks.

In [ ]:
# The three key numbers

# Number 1: Pages with good position but low CTR
low_ctr_opportunity = visible[(visible['avg_position'] <= 20) & (visible['ctr'] < 1)]
print(f'1. Pages with position <= 20 but CTR < 1%: {len(low_ctr_opportunity):,}')

# Number 2: Total impressions flowing through these pages
total_impressions_at_risk = low_ctr_opportunity['impressions_90d'].sum()
print(f'2. Total impressions through low-CTR pages: {total_impressions_at_risk:,.0f}')

# Number 3: Highest-value page-1 candidates
page1_low_ctr = visible[(visible['avg_position'] <= 10) & (visible['ctr'] < 0.5)]
print(f'3. Page-1 pages (position <= 10) with CTR < 0.5%: {len(page1_low_ctr):,}')

print(f'\nThese page-1 pages average {page1_low_ctr["impressions_90d"].mean():,.0f} impressions each')
print(f'Current clicks from these pages: {page1_low_ctr["clicks_90d"].sum():,.0f}')
print(f'If CTR doubled, additional clicks: ~{int(page1_low_ctr["impressions_90d"].sum() * page1_low_ctr["ctr"].mean() / 100):,}')

## 4. Careful words: what I can and can't claim

**What I can claim (observational, directional, decision-support):**
- I can say: "We observed that X pages with position ≤ 20 have CTR below 1%, suggesting an opportunity for snippet improvement."
- I can say: "Pages in position tier Y tend to have lower CTR than expected, which may indicate title/meta description issues."
- I can say: "Our model ranks pages by likelihood of CTR improvement potential, helping teams prioritize their review time."

**What I cannot claim (causal, predictive, or overreaching):**
- I cannot claim: "Rewriting titles WILL increase CTR" — that requires an experiment with before/after measurement.
- I cannot claim: "Google's algorithm favors X" — I'm observing correlations, not ranking factors.
- I cannot claim: "This model predicts CTR" — it ranks pages by opportunity, not by future CTR values.
- I cannot claim: "AI-referred traffic is growing" — the data shows click-throughs from AI tools, not citations or visibility.

The work is decision-support: it helps humans spend their limited time on the pages most likely to benefit from review.

In [ ]:
# Final summary: what this data tells us

print('=== SUMMARY: WHY CTR / ENGAGEMENT OPPORTUNITY SCORING ===')
print(f'\nDataset: {len(df):,} pages across {df["client_id"].nunique()} clients')
print(f'Visible pages (>= 100 impressions): {len(visible):,} ({len(visible)/len(df)*100:.1f}%)')
print(f'\nKey opportunity metrics:')
print(f'  - {len(low_ctr_opportunity):,} pages with good position but low CTR')
print(f'  - {total_impressions_at_risk:,.0f} total impressions at stake')
print(f'  - {len(page1_low_ctr):,} highest-value page-1 candidates')
print(f'\nThis is not just "train a model" — it is:')
print(f'  1. A decision-support tool for content teams')
print(f'  2. Built on observable signals (CTR, position, impressions)')
print(f'  3. Designed to beat a transparent baseline rule')
print(f'  4. Validated with client-holdout to avoid overfitting')
print(f'  5. Framed with careful, non-causal language')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.